In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_party_whl(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "PartyID string"
        #     ",SourceSystem string"
        #     ",PortfolioKey string"
        #     ",PortfolioName string"
        #     ",IngestionTimestamp timestamp"
        #     ",BatchId string"
        #     ",CDI string"
        #     ",RepresentedByPartyKey string"
        #     ",RepresentedByPartyName string"
        #     ",RepresentedByPartyType string"
        #     ",PartyKey string"
        #     ",PartyRepresentKey string"
        #     ",PartyTypeCode string"
        #     ",PartyType string"
        #     ",PartyStatusCode string"
        #     ",PartyStatus string"
        #     ",PartyStatusDate date"
        #     ",L3869StatusCode string"
        #     ",L3869Status string"
        #     ",L3869CourtDate timestamp"
        #     ",OCWStatus string"
        #     ",DeceasedFlag boolean"
        #     ",L3869InterbankingStatusCode int"
        #     ",L3869InterBankingStatus string"
        #     ",Watchlist string"
        #     ",FirstNameGR string"
        #     ",FirstNameENG string"
        #     ",LastNameGR string"
        #     ",LastNameENG string"
        #     ",FirstFatherNameGR string"
        #     ",FirstFatherNameENG string"
        #     ",LastFatherNameGR string"
        #     ",LastFatherNameENG string"
        #     ",FirstMotherNameGR string"
        #     ",LastMotherNameGR string"
        #     ",LastMotherNameENG string"
        #     ",LegalNameGR string"
        #     ",LegalNameENG string"
        #     ",PrimaryCommercialGR string"
        #     ",PrimaryCommercialENG string"
        #     ",SalutationTypeCode string"
        #     ",SalutationType string"
        #     ",GenderCode string"
        #     ",Gender string"
        #     ",MaritalStatusCode string"
        #     ",MaritalStatus string"
        #     ",CitizenshipCode string"
        #     ",Citizenship string"
        #     ",NationalityCode string"
        #     ",Nationality string"
        #     ",OccupationCode string"
        #     ",Occupation string"
        #     ",BirthDate string"
        #     ",BirthPlace string"
        #     ",DeathDate date"
        #     ",CommunicationLanguageCode int"
        #     ",CommunicationLanguage string"
        #     ",TaxRegistrationNumber string"
        #     ",TaxResidenceCountry string"
        #     ",OrganizationOriginCountry string"
        #     ",PassportNumber string"
        #     ",PassportIssuingAuthority string"
        #     ",PassportIssueDate date"
        #     ",PassportExpiredDate date"
        #     ",NationalNumber string"
        #     ",NationalNumberIssuingAuthority string"
        #     ",NationalNumberIssueDate date"
        #     ",NationalNumberExpiredDate date"
        #     ",MilitaryNumber string"
        #     ",MilitaryAuthority string"
        #     ",MilitaryIssueDate date"
        #     ",MilitaryExpiredDate date"
        #     ",SocialSecurityNumber string"
        #     ",ChildrenCertification string"
        #     ",ChildrenCertificationIssuingAuthority string"
        #     ",EnterpriseStartDate string"
        #     ",EnterpriseEndDate date"
        #     ",EconomicActivityCode int"
        #     ",EconomicActivity string"
        #     ",BranchActivityCode int"
        #     ",BranchActivity string"
        #     ",NumberOfChildren string"
        #     ",GeneralCommercialRegister string"
        #     ",NumberofEmployees string"
        #     ",TotalIncome string"
        #     ",BalanceSheetTotalAmount string"
        #     ",AnnualTurnoverAmount string"
        #     ",GroupCode string"
        #     ",GroupName string"
        #     ",GroupCodePortfolio int"
        #     ",PartyBankruptcyStage int"
        #     ",CovidFlag string"
        #     ",CovidCategoryCode int"
        #     ",CovidCategory string"
        #     ",AffectedCategoryCode int"
        #     ",AffectedCategory string"
        #     ",MonitoringUnitCode string"
        #     ",MonitoringUnit string"
        #     ",CreditRatingTypeCode int"
        #     ",CreditRatingType string"
        #     ",EducationLevelCode int"
        #     ",EducationLevel string"
        #     ",BalanceCategoryCode string"
        #     ",BalanceCategory string"
        #     ",ExtraSegmentCode string"
        #     ",ExtraSegment string"
        #     ",SegmentCode string"
        #     ",Segment string"
        #     ",SubSegmentCode string"
        #     ",SubSegment string"
        #     ",LegalFormCode int"
        #     ",LegalForm string"
        #     ",IsWholeshaleFlag boolean"
        #     ",EmploymentStatusCode int"
        #     ",EmploymentStatus string"
        #     ",EmploymentStatusDate date"
        #     ",CreatedBy string"
        #     ",CreatedByDate timestamp"
        #     ",LastUpdatedBy string"
        #     ",LastUpdatedByDate timestamp"
        #     ",ResubmissionStatus string"
        #     ",N4605Flag int"
        #     ",ConnectedBorrowerFlag boolean"
        #     ",StaffFlag string"
        #     ",L3869BorrowerHasOver50KBalance boolean"
        #     ",TargetLegalActionCode int"
        #     ",TargetLegalAction string"
        #     ", SubServicer string"
        #     ", RiskDescription string"
        #     ",MobileTel string"
        #     ",MobileTel2 string"
        #     ",BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Party"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "whl",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== additionaldebtorsdemographics ==================

        additionaldebtorsdemographics_extra_filters = None

        if additionaldebtorsdemographics_extra_filters is not None:
            additionaldebtorsdemographics_extra_filters.append(date_filter)
            additionaldebtorsdemographics_final_filters = " AND ".join(
                additionaldebtorsdemographics_extra_filters
            )
        else:
            additionaldebtorsdemographics_final_filters = date_filter

            additionaldebtorsdemographics_selected_fields = [
                "adddemogrlegalstatusid",
                "adddemogrcreateuser",
                "adddemogrcreatedate",
                "adddemogrmodifieduser",
                "adddemogrlastmodifieddate",
                "adddemogrcdi",
                "businessdate",
            ]

            additionaldebtorsdemographics = load_landed_delta_table(
                spark,
                catalog_landed,  # type: ignore
                "whl_additionaldebtorsdemographics",
                additionaldebtorsdemographics_final_filters,
                additionaldebtorsdemographics_selected_fields,
            )

        # ================== debtorstatus ==================

        debtorstatus_extra_filters = None

        if debtorstatus_extra_filters is not None:
            debtorstatus_extra_filters.append(date_filter)
            debtorstatus_final_filters = " AND ".join(debtorstatus_extra_filters)
        else:
            debtorstatus_final_filters = date_filter

            debtorstatus_selected_fields = [
                "debtorstatusid",
                "portfolioid",
                "debtorstatusprfcdi",
                "debtorstatusprfcustomername",
                "debtorstatusprfafm",
                "debtorstatusprfgroupid",
                "debtorstatusprfgroupdescription",
                "businessdate",
            ]

            debtorstatus = load_landed_delta_table(
                spark,
                catalog_landed,  # type: ignore
                "whl_debtorstatus",
                debtorstatus_final_filters,
                debtorstatus_selected_fields,
            )

        # ================== rfdtlegalstatus ==================

        rfdtlegalstatus_extra_filters = None

        if rfdtlegalstatus_extra_filters is not None:
            rfdtlegalstatus_extra_filters.append(date_filter)
            rfdtlegalstatus_final_filters = " AND ".join(rfdtlegalstatus_extra_filters)
        else:
            rfdtlegalstatus_final_filters = date_filter

            rfdtlegalstatus_selected_fields = [
                "description",
                "id",
                "businessdate",
            ]

            rfdtlegalstatus = load_landed_delta_table(
                spark,
                catalog_landed,  # type: ignore
                "whl_rfdtlegalstatus",
                rfdtlegalstatus_final_filters,
                rfdtlegalstatus_selected_fields,
            )

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "portfolioname",
            "portfolio_id_optiverse",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ====================== party_whl ==============

        party_whl_primary_key_fields = [
            debtorstatus.debtorstatusid,
            lit("whl"),
        ]

        party_whl = (
            debtorstatus.join(
                additionaldebtorsdemographics,
                (
                    additionaldebtorsdemographics["adddemogrcdi"]
                    == debtorstatus["debtorstatusprfcdi"]
                )
                & (
                    additionaldebtorsdemographics["businessdate"]
                    == debtorstatus["businessdate"]
                ),
                "left",
            )
            .join(
                rfdtlegalstatus,
                (
                    additionaldebtorsdemographics["adddemogrlegalstatusid"]
                    == rfdtlegalstatus["id"]
                )
                & (
                    additionaldebtorsdemographics["businessdate"]
                    == rfdtlegalstatus["businessdate"]
                ),
                "left",
            )
            .join(
                portfolio,
                debtorstatus.portfolioid == portfolio.portfolio_id_optiverse,
                "left",
            )
            .withColumn(
                "PartyID", sha2(concat_ws(";", *party_whl_primary_key_fields), 256)
            )
            .withColumn(
                "DeceasedFlag",
                when(
                    (additionaldebtorsdemographics.adddemogrlegalstatusid == 5),
                    lit(True),
                ).otherwise(lit(False)),
            )
        )

        # ================== Selection of Final Columns ==================

        party_whl = party_whl.select(
            party_whl.PartyID.alias("PartyID"),
            lit("whl").cast("string").alias("SourceSystem"),
            debtorstatus.portfolioid.cast("string").alias("PortfolioKey"),
            portfolio.portfolioname.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("string").alias("BatchId"),
            party_whl.debtorstatusprfcdi.cast("string").alias("CDI"),
            lit(None).cast("string").alias("RepresentedByPartyKey"),
            lit(None).cast("string").alias("RepresentedByPartyName"),
            lit(None).cast("string").alias("RepresentedByPartyType"),
            party_whl.debtorstatusid.cast("string").alias("PartyKey"),
            lit(None).cast("string").alias("PartyRepresentKey"),
            lit("2").cast("string").alias("PartyTypeCode"),
            lit("Organization").cast("string").alias("PartyType"),
            party_whl.adddemogrlegalstatusid.cast("string").alias("PartyStatusCode"),
            party_whl.description.cast("string").alias("PartyStatus"),
            lit(None).cast("date").alias("PartyStatusDate"),
            lit(None).cast("string").alias("L3869StatusCode"),
            lit(None).cast("string").alias("L3869Status"),
            lit(None).cast("timestamp").alias("L3869CourtDate"),
            lit(None).alias("OCWStatus"),
            party_whl.DeceasedFlag.cast("boolean").alias("DeceasedFlag"),
            lit(None).cast("int").alias("L3869InterbankingStatusCode"),
            lit(None).cast("string").alias("L3869InterbankingStatus"),
            lit(None).alias("Watchlist"),
            party_whl.debtorstatusprfcustomername.cast("string").alias("FirstNameGR"),
            lit(None).cast("string").alias("FirstNameENG"),
            party_whl.debtorstatusprfcustomername.cast("string").alias("LastNameGR"),
            lit(None).cast("string").alias("LastNameENG"),
            lit(None).cast("string").alias("FirstFatherNameGR"),
            lit(None).cast("string").alias("FirstFatherNameENG"),
            lit(None).cast("string").alias("LastFatherNameGR"),
            lit(None).cast("string").alias("LastFatherNameENG"),
            lit(None).cast("string").alias("FirstMotherNameGR"),
            lit(None).cast("string").alias("LastMotherNameGR"),
            lit(None).cast("string").alias("LastMotherNameENG"),
            party_whl.debtorstatusprfcustomername.cast("string").alias("LegalNameGR"),
            lit(None).cast("string").alias("LegalNameENG"),
            party_whl.debtorstatusprfcustomername.alias("PrimaryCommercialGR"),
            lit(None).cast("string").alias("PrimaryCommercialENG"),
            lit(None).cast("string").alias("SalutationTypeCode"),
            lit(None).cast("string").alias("SalutationType"),
            lit(None).cast("string").alias("GenderCode"),
            lit(None).cast("string").alias("Gender"),
            lit(None).cast("string").alias("MaritalStatusCode"),
            lit(None).cast("string").alias("MaritalStatus"),
            lit(None).alias("CitizenshipCode"),
            lit(None).cast("string").alias("Citizenship"),
            lit(None).cast("string").alias("NationalityCode"),
            lit(None).cast("string").alias("Nationality"),
            lit(None).cast("string").alias("OccupationCode"),
            lit(None).cast("string").alias("Occupation"),
            lit(None).cast("string").alias("BirthDate"),
            lit(None).cast("string").alias("BirthPlace"),
            lit(None).cast("date").alias("DeathDate"),
            lit(None).alias("CommunicationLanguageCode"),
            lit(None).cast("string").alias("CommunicationLanguage"),
            party_whl.debtorstatusprfafm.cast("string").alias("TaxRegistrationNumber"),
            lit(None).cast("string").alias("TaxResidenceCountry"),
            lit(None).cast("string").alias("OrganizationOriginCountry"),
            lit(None).cast("string").alias("PassportNumber"),
            lit(None).cast("string").alias("PassportIssuingAuthority"),
            lit(None).cast("date").alias("PassportIssueDate"),
            lit(None).cast("date").alias("PassportExpiredDate"),
            lit(None).cast("string").alias("NationalNumber"),
            lit(None).cast("string").alias("NationalNumberIssuingAuthority"),
            lit(None).cast("date").alias("NationalNumberIssueDate"),
            lit(None).cast("date").alias("NationalNumberExpiredDate"),
            lit(None).cast("string").alias("MilitaryNumber"),
            lit(None).cast("string").alias("MilitaryAuthority"),
            lit(None).cast("date").alias("MilitaryIssueDate"),
            lit(None).cast("date").alias("MilitaryExpiredDate"),
            lit(None).cast("string").alias("SocialSecurityNumber"),
            lit(None).cast("string").alias("ChildrenCertification"),
            lit(None).cast("string").alias("ChildrenCertificationIssuingAuthority"),
            lit(None).cast("string").alias("EnterpriseStartDate"),
            lit(None).cast("date").alias("EnterpriseEndDate"),
            lit(None).cast("int").alias("EconomicActivityCode"),
            lit(None).cast("string").alias("EconomicActivity"),
            lit(None).cast("int").alias("BranchActivityCode"),
            lit(None).cast("string").alias("BranchActivity"),
            lit(None).cast("string").alias("NumberOfChildren"),
            lit(None).cast("string").alias("GeneralCommercialRegister"),
            lit(None).cast("string").alias("NumberofEmployees"),
            lit(None).cast("string").alias("TotalIncome"),
            lit(None).cast("string").alias("BalanceSheetTotalAmount"),
            lit(None).cast("string").alias("AnnualTurnoverAmount"),
            party_whl.debtorstatusprfgroupid.cast("string").alias("GroupCode"),
            party_whl.debtorstatusprfgroupdescription.cast("string").alias("GroupName"),
            lit(None).alias("GroupCodePortfolio"),
            lit(None).cast("int").alias("PartyBankruptcyStage"),
            lit(None).cast("string").alias("CovidFlag"),
            lit(None).cast("int").alias("CovidCategoryCode"),
            lit(None).cast("string").alias("CovidCategory"),
            lit(None).alias("AffectedCategoryCode"),
            lit(None).alias("AffectedCategory"),
            lit(None).cast("string").alias("MonitoringUnitCode"),
            lit(None).cast("string").alias("MonitoringUnit"),
            lit(None).cast("int").alias("CreditRatingTypeCode"),
            lit(None).cast("string").alias("CreditRatingType"),
            lit(None).cast("int").alias("EducationLevelCode"),
            lit(None).cast("string").alias("EducationLevel"),
            lit(None).cast("string").alias("BalanceCategoryCode"),
            lit(None).cast("string").alias("BalanceCategory"),
            lit(None).cast("string").alias("ExtraSegmentCode"),
            lit(None).cast("string").alias("ExtraSegment"),
            lit(None).cast("string").alias("SegmentCode"),
            lit(None).cast("string").alias("Segment"),
            lit(None).cast("string").alias("SubSegmentCode"),
            lit(None).cast("string").alias("SubSegment"),
            lit(None).cast("int").alias("LegalFormCode"),
            lit(None).cast("string").alias("LegalForm"),
            lit(True).alias("IsWholeshaleFlag"),
            lit(None).alias("EmploymentStatusCode"),
            lit(None).cast("string").alias("EmploymentStatus"),
            lit(None).cast("date").alias("EmploymentStatusDate"),
            party_whl.adddemogrcreateuser.cast("string").alias("CreatedBy"),
            party_whl.adddemogrcreatedate.cast("timestamp").alias("CreatedByDate"),
            party_whl.adddemogrmodifieduser.cast("string").alias("LastUpdatedBy"),
            party_whl.adddemogrlastmodifieddate.cast("timestamp").alias(
                "LastUpdatedByDate"
            ),
            lit(None).alias("ResubmissionStatus"),
            lit(None).alias("N4605Flag"),
            lit(None).alias("ConnectedBorrowerFlag"),
            lit(None).alias("StaffFlag"),
            lit(None).alias("L3869BorrowerHasOver50KBalance"),
            lit(None).alias("TargetLegalActionCode"),
            lit(None).alias("TargetLegalAction"),
            lit(None).cast("string").alias("SubServicer"),
            lit(None).cast("string").alias("RiskDescription"),
            lit(None).cast("string").alias("MobileTel"),
            lit(None).cast("string").alias("MobileTel2"),
            debtorstatus.businessdate.cast("date").alias("BusinessDate"),
        )

        # ================== Append ==================

        party_whl.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_party_whl(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)